# 00 — Environment check

**Run this first.** It touches no models and downloads nothing. It answers one
question: will the training notebooks actually work on this machine?

Every check prints an explicit `PASS` / `FAIL` / `WARN`. If anything says FAIL,
fix it before running notebooks 01–03 — a failure here becomes a confusing
crash forty minutes into training otherwise.

**Hardware this was written for:** single NVIDIA RTX 5060 Ti, 16 GB VRAM.

## 1. GPU and CUDA

The RTX 5060 Ti is a **Blackwell** card (compute capability `sm_120`). This
matters more than it sounds: PyTorch wheels built for CUDA 12.1 or 12.4 will
import cleanly, report `cuda.is_available() == True`, and then die at the first
real kernel launch with *"no kernel image is available for execution on the
device"*.

The cell below therefore does not stop at `is_available()` — it runs an actual
matrix multiply on the GPU, which is the only check that catches this.

In [1]:
import sys, platform

print(f"Python   : {sys.version.split()[0]}")
print(f"Platform : {platform.platform()}")
print()

results = {}

try:
    import torch
    print(f"torch    : {torch.__version__}")
    print(f"CUDA build against: {torch.version.cuda}")

    if not torch.cuda.is_available():
        print("\nFAIL  No CUDA device visible to PyTorch.")
        print("      Check your driver, then reinstall torch from the cu128 index")
        print("      (see the pinned-versions cell below).")
        results["gpu"] = False
    else:
        name = torch.cuda.get_device_name(0)
        cap = torch.cuda.get_device_capability(0)
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"\nDevice   : {name}")
        print(f"Capability: sm_{cap[0]}{cap[1]}")
        print(f"Total VRAM: {total:.2f} GB")

        # The real test: does a kernel actually launch?
        try:
            x = torch.randn(512, 512, device="cuda", dtype=torch.bfloat16)
            _ = (x @ x).sum().item()
            torch.cuda.synchronize()
            print("\nPASS  A bf16 matmul ran on the GPU. Kernels are working.")
            results["gpu"] = True
        except Exception as e:
            print(f"\nFAIL  CUDA is 'available' but kernels do not run: {e}")
            print("      This is the classic sm_120 / wrong-CUDA-wheel symptom.")
            print("      Reinstall torch from the cu128 index.")
            results["gpu"] = False

        if total < 15:
            print(f"\nWARN  {total:.1f} GB of VRAM detected; these notebooks assume ~16 GB.")

        free, _ = torch.cuda.mem_get_info()
        print(f"\nCurrently free: {free/1e9:.2f} GB")
        if free / 1e9 < 12:
            print("WARN  Something else is holding VRAM. Close other GPU processes")
            print("      (browsers and games are the usual culprits) before training.")
except ImportError:
    print("FAIL  PyTorch is not installed.")
    results["gpu"] = False

Python   : 3.11.16
Platform : Windows-10-10.0.26100-SP0

torch    : 2.7.0+cu128
CUDA build against: 12.8

Device   : NVIDIA GeForce RTX 5060 Ti
Capability: sm_120
Total VRAM: 17.10 GB

PASS  A bf16 matmul ran on the GPU. Kernels are working.

Currently free: 15.84 GB


## 2. Required packages

In [2]:
import importlib.metadata as md_

REQUIRED = {
    "torch":        "2.7.0",
    "transformers": "4.44.0",
    "peft":         "0.12.0",
    "accelerate":   "0.33.0",
    "datasets":     "2.20.0",
    "safetensors":  "0.4.3",
}
OPTIONAL = {"pandas": "2.0.0", "matplotlib": "3.7.0", "scipy": "1.11.0"}

def parse(v):
    out = []
    for part in v.split(".")[:3]:
        num = "".join(ch for ch in part if ch.isdigit())
        out.append(int(num) if num else 0)
    return tuple(out)

print(f"{'package':<15} {'installed':<14} {'minimum':<10} status")
print("-" * 55)

all_ok = True
for pkg, minimum in REQUIRED.items():
    try:
        got = md_.version(pkg)
        ok = parse(got) >= parse(minimum)
        print(f"{pkg:<15} {got:<14} {minimum:<10} {'PASS' if ok else 'FAIL (too old)'}")
        all_ok &= ok
    except md_.PackageNotFoundError:
        print(f"{pkg:<15} {'-':<14} {minimum:<10} FAIL (not installed)")
        all_ok = False

print()
for pkg, minimum in OPTIONAL.items():
    try:
        got = md_.version(pkg)
        print(f"{pkg:<15} {got:<14} {minimum:<10} (optional)")
    except md_.PackageNotFoundError:
        print(f"{pkg:<15} {'-':<14} {minimum:<10} WARN (optional; notebook 05 uses pandas)")

results["packages"] = all_ok
print("\n" + ("PASS  All required packages present." if all_ok
               else "FAIL  Install the missing packages (see the next cell)."))

package         installed      minimum    status
-------------------------------------------------------
torch           2.7.0+cu128    2.7.0      PASS
transformers    4.50.0         4.44.0     PASS
peft            0.13.2         0.12.0     PASS
accelerate      0.34.2         0.33.0     PASS
datasets        2.21.0         2.20.0     PASS
safetensors     0.4.5          0.4.3      PASS

pandas          2.2.3          2.0.0      (optional)
matplotlib      3.9.2          3.7.0      (optional)
scipy           1.14.1         1.11.0     (optional)

PASS  All required packages present.


### Pinned install commands

Install **torch first, from the cu128 index**, then everything else. Installing
them together lets pip resolve a default-CUDA torch wheel that will not run on
this card.

```bash
python -m venv .venv
.venv\Scripts\activate            # Windows
# source .venv/bin/activate        # Linux / macOS

pip install --upgrade pip

# 1. torch, from the Blackwell-compatible index
pip install torch==2.7.0 --index-url https://download.pytorch.org/whl/cu128

# 2. everything else
pip install \
  transformers==4.44.2 \
  peft==0.13.2 \
  accelerate==0.34.2 \
  datasets==2.21.0 \
  safetensors==0.4.5 \
  sentencepiece==0.2.0 \
  pandas==2.2.3 \
  scipy==1.14.1 \
  matplotlib==3.9.2 \
  ipywidgets==8.1.5
```

`ipywidgets` is only there to stop the HuggingFace progress bars from spamming
plain text into your notebook output.

**No `bitsandbytes`.** You asked for standard LoRA on bf16 base weights, not
QLoRA, so nothing here needs 4-bit quantization support.

## 3. Model repositories

Metadata only — this reads the model cards, it does **not** download weights.
It also reports each model's `model_type` and tokenizer vocabulary size, both of
which matter for the comparison.

In [3]:
MODEL_IDS = {
    "tigerllm_1b": "md-nishat-008/TigerLLM-1B-it",
    "titullm_1b": "hishab/titulm-llama-3.2-1b-v2.0",
    "titullm_3b": "hishab/titulm-llama-3.2-3b-v2.0",
}

try:
    from huggingface_hub import HfApi
    api = HfApi()
    ok = True
    for key, rid in MODEL_IDS.items():
        try:
            info = api.model_info(rid)
            cfg = info.config or {}
            mtype = cfg.get("model_type", "?")
            arch = cfg.get("architectures", ["?"])
            print(f"PASS  {key:<12} {rid}")
            print(f"      model_type={mtype}  architectures={arch}")
            print(f"      revision={info.sha[:12] if info.sha else '?'}   "
                  f"gated={bool(getattr(info, 'gated', False))}")
            print()
        except Exception as e:
            print(f"FAIL  {key:<12} {rid}\n      {str(e)[:180]}\n")
            ok = False
    results["models"] = ok
except ImportError:
    print("WARN  huggingface_hub not installed; skipping the repo check.")
    results["models"] = None

PASS  tigerllm_1b  md-nishat-008/TigerLLM-1B-it
      model_type=gemma3_text  architectures=['Gemma3ForCausalLM']
      revision=bc84e6d99559   gated=False

PASS  titullm_1b   hishab/titulm-llama-3.2-1b-v2.0
      model_type=llama  architectures=['LlamaForCausalLM']
      revision=cf9ebaf81f39   gated=False

PASS  titullm_3b   hishab/titulm-llama-3.2-3b-v2.0
      model_type=llama  architectures=['LlamaForCausalLM']
      revision=ce14af760eff   gated=False



### Architecture note — worth reading before you train

`TigerLLM-1B`'s paper describes it as continual pretraining on top of
**Llama-3.2-1B**, but the Hugging Face repo has at times been tagged as a
**Gemma-3** family model. The cell above prints the real `model_type`.

Why this matters for RQ1: **H1a only isolates pretraining depth if both 1B
models share an architecture.** If TigerLLM-1B turns out to be Gemma-3 based
while TituLLM-1B is Llama-3.2 based, then architecture varies alongside depth
and the comparison cannot cleanly attribute a gap to Bengali pretraining alone.

If they differ, you do not have to abandon the experiment — but the claim has to
soften from *"we isolate pretraining depth"* to *"we compare two 1B Bengali
models that differ in both architecture and pretraining depth."* Note whatever
the cell prints and carry it into your limitations section.

One practical consequence: LoRA target modules differ between the two families.
Llama uses `q_proj/k_proj/v_proj/o_proj/gate_proj/up_proj/down_proj`; Gemma-3
uses the same names, so the target list in notebooks 01–03 works either way.
The tokenizers, however, are completely different.

## 4. Tokenizer quirks

Three things that bite here, all handled in the training notebooks:

1. **No pad token.** Llama-family tokenizers ship without one. Every notebook
   sets `pad_token = eos_token`. Without it the collator throws.
2. **Extended vocabulary.** The TituLM v2.0 models extend Llama-3.2's 128K
   vocabulary to roughly 170K to fit Bengali more efficiently. That inflates the
   embedding and LM-head matrices — which is why TituLLM-1B has more parameters
   than a stock Llama-3.2-1B. LoRA does not touch those layers, so it costs
   memory but not trainable parameters.
3. **`trust_remote_code`.** All three are standard architectures served by
   `transformers`, so this should **not** be needed. The cell below checks
   rather than assuming, and the training notebooks expose a flag if it is.

In [4]:
try:
    from transformers import AutoTokenizer, AutoConfig
    for key, rid in MODEL_IDS.items():
        try:
            cfg = AutoConfig.from_pretrained(rid)
            tok = AutoTokenizer.from_pretrained(rid)
            needs_trc = getattr(cfg, "auto_map", None) is not None
            print(f"{key:<12} vocab={len(tok):<8} model_type={cfg.model_type:<14} "
                  f"pad={'set' if tok.pad_token else 'MISSING -> will use eos'}")
            print(f"             trust_remote_code needed: {needs_trc}")
            if needs_trc:
                print("             ^ set TRUST_REMOTE_CODE = True in the training notebook")
        except Exception as e:
            print(f"{key:<12} could not load tokenizer/config: {str(e)[:140]}")
        print()
except ImportError:
    print("WARN  transformers not installed; skipping tokenizer checks.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

C:\anaconda\envs\bengali-rq1\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC\.cache\huggingface\hub\models--md-nishat-008--TigerLLM-1B-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP down

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tigerllm_1b  vocab=262145   model_type=gemma3_text    pad=set
             trust_remote_code needed: False

titullm_1b   vocab=170497   model_type=llama          pad=set
             trust_remote_code needed: False

titullm_3b   vocab=170497   model_type=llama          pad=set
             trust_remote_code needed: False



## 5. Disk space

In [9]:
import shutil, os
from pathlib import Path

free_gb = shutil.disk_usage(Path.cwd()).free / 1e9
print(f"Free space on this drive: {free_gb:.1f} GB")
print()
print("Rough requirements:")
print("  TigerLLM-1B weights   ~2.5 GB")
print("  TituLLM-1B weights    ~2.5 GB")
print("  TituLLM-3B weights    ~6.5 GB")
print("  HF cache overhead     ~2 GB")
print("  LoRA adapters         ~20-60 MB each (tiny)")
print("  ----------------------------")
print("  Total                 ~15 GB, plus room for eval outputs")
print()
if free_gb < 30:
    print("WARN  Under 30 GB free. Tight but probably workable.")
else:
    print("PASS  Plenty of space.")

cache = os.environ.get("HF_HOME") or os.environ.get("HF_HUB_CACHE")
print(f"\nHF cache location: {cache or 'default (~/.cache/huggingface)'}")
print("Set HF_HOME to another drive if your system drive is nearly full.")

Free space on this drive: 299.4 GB

Rough requirements:
  TigerLLM-1B weights   ~2.5 GB
  TituLLM-1B weights    ~2.5 GB
  TituLLM-3B weights    ~6.5 GB
  HF cache overhead     ~2 GB
  LoRA adapters         ~20-60 MB each (tiny)
  ----------------------------
  Total                 ~15 GB, plus room for eval outputs

PASS  Plenty of space.

HF cache location: default (~/.cache/huggingface)
Set HF_HOME to another drive if your system drive is nearly full.


## 6. Summary

In [16]:
print("=" * 58)
print("ENVIRONMENT CHECK SUMMARY")
print("=" * 58)
labels = {"gpu": "GPU + CUDA kernels", "packages": "Required packages",
          "models": "Model repos reachable"}
for k, label in labels.items():
    v = results.get(k)
    status = "PASS" if v is True else ("FAIL" if v is False else "SKIPPED")
    print(f"  {label:<24} {status}")

if all(results.get(k) is not False for k in labels):
    print("\nReady. Next: 01_train_tigerllm_1b.ipynb")
else:
    print("\nFix the FAIL items above before training. Do not skip ahead —")
    print("a broken environment shows up as a crash mid-run, not as a clear error.")

ENVIRONMENT CHECK SUMMARY
  GPU + CUDA kernels       PASS
  Required packages        PASS
  Model repos reachable    PASS

Ready. Next: 01_train_tigerllm_1b.ipynb


> ### ⚠️ Version mismatch worth a second look
>
> Your model list pairs **`titulm-llama-3.2-1b-v1.1`** with
> **`titulm-llama-3.2-3b-v2.0`**. These are different generations of the TituLM
> family, not the same recipe at two sizes:
>
> | Repo | Generation |
> |---|---|
> | `hishab/titulm-llama-3.2-1b-v1.1` | v1.x series (older release) |
> | `hishab/titulm-llama-3.2-3b-v2.0` | v2.0 series (later release, retrained corpus + extended tokenizer) |
>
> H1b claims to isolate **scale** by holding the pretraining approach constant.
> With a v1.1 vs v2.0 pair, corpus and tokenizer changes ride along with the
> size difference, so a reviewer can argue the gap is not purely scale.
>
> **`hishab/titulm-llama-3.2-1b-v2.0` exists** if you want a version-matched
> pair. I have not changed your IDs — this is your call. If you keep v1.1,
> say so explicitly in the paper's limitations.